# 1.4 Multimodal Messages — Text, Images, and Audio

## What you will learn in this notebook

- How LangChain's **content list format** lets you mix text, images, and audio in one message
- How to **upload an image** in a notebook and send it to a vision model
- How to **record audio** from your microphone and send it to an audio model
- Why **base64 encoding** is used to send binary data through a text-based API
- Which **model variants** support which modalities

---

### What does "multimodal" mean?

**Modality** = a type of input/output (text, image, audio, video).

Early LLMs only understood **text**. Modern models like GPT-4o and Claude 3 are **multimodal** — they can read text AND see images AND hear audio.

```
Text-only:      "Describe this image"  →  ❌ (model can't see it)
Multimodal:     [image + "Describe this image"]  →  ✓
```

### How does it work technically?

LangChain represents a multimodal message as a **list of content blocks** instead of a plain string:

```python
HumanMessage(content=[
    {"type": "text",  "text": "Tell me about this"},
    {"type": "image", "base64": "...", "mime_type": "image/png"},
])
```

Each block has a `type` field. The model receives all blocks together and reasons across them simultaneously.

### Why base64?

APIs communicate over HTTP using **text** (JSON). Images and audio are **binary** files. **Base64** is an encoding that converts any binary data into a safe ASCII string that can travel inside a JSON payload without corruption. The model provider decodes it back to binary on their end.

```
PNG file (binary)  →  base64.b64encode()  →  "iVBORw0KGgoAAAA..."  →  JSON payload
```

---

### Model support matrix

| Model | Text | Images | Audio |
|---|---|---|---|
| `gpt-5-nano` | ✓ | ✓ | ✗ |
| `gpt-4o` | ✓ | ✓ | ✗ |
| `gpt-4o-audio-preview` | ✓ | ✓ | ✓ |
| `claude-sonnet-4-5` | ✓ | ✓ | ✗ |

Full model list: https://platform.openai.com/docs/models

---
## Section 1 — Text Input (Content List Format)

In [1]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================

from dotenv import load_dotenv

load_dotenv()

True

In [39]:
# ============================================================
# CELL 2: Create the Sci-Fi Agent
# ============================================================
# Same agent from the prompting notebook — we reuse it here
# to keep the context consistent across examples.

from langchain.agents import create_agent

agent = create_agent(
    model='groq:qwen/qwen3.6-27b',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [40]:
# ============================================================
# CELL 3: Text Input Using the Content List Format
# ============================================================
# Previously we passed a plain string to HumanMessage:
#   HumanMessage(content="What is the capital of The Moon?")
#
# Now we pass a LIST of content blocks:
#   HumanMessage(content=[{"type": "text", "text": "..."}])
#
# Both are equivalent for text-only messages, but the list format
# is the REQUIRED format when mixing modalities (text + image, etc.).
# Learning it now as a text-only example prepares you for images
# and audio in the next sections.
#
# Content block structure:
#   {"type": "text", "text": "<your message>"}  ← for plain text
#   {"type": "image", "base64": "...", "mime_type": "image/png"}  ← for images
#   {"type": "audio", "base64": "...", "mime_type": "audio/wav"}  ← for audio

from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
    # A list with one text block — same as passing a plain string
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Role:** Science fiction writer
   - **Task:** Create a capital city at the user's request
   - **Question:** "What is the capital of The Moon?"

2.  **Identify Key Constraints & Requirements:**
   - The user is asking for the capital of "The Moon"
   - As an AI, I know the Moon has no real capital (it's not a sovereign state)
   - But my role is to act as a science fiction writer and create one per the request
   - I need to invent a fictional, compelling capital city for the Moon, complete with sci-fi elements, worldbuilding, and atmospheric details

3.  **Brainstorming Core Concepts:**
   - Name: Needs to sound lunar, futuristic, maybe mythological or scientific
     - Ideas: Selene Prime, Artemis Reach, Mare Tranquillitatis Station, Luna Central, Crateris, Silentium, Argentis, Mare Serenitatis Capital, Lunaris, Palimpsest, Cineris, Aethelgard (no, too fantasy), Novus Orbis, Selene-7, The Silver Spire, Mare Imbri

---
## Section 2 — Image Input

To send an image to the model, we need to:
1. Get the image as **bytes** (from a file, upload widget, camera, etc.)
2. **Base64 encode** the bytes into a string
3. Include the base64 string in a content block with `type: "image"`

The notebook upload widget (`FileUpload`) gives us a clean way to do this without dealing with file paths.

In [41]:
# ============================================================
# CELL 4: Display an Image Upload Widget
# ============================================================
# FileUpload is a Jupyter widget that renders a file picker button
# directly in the notebook output.
#
# Parameters:
#   accept='.png'   → only show PNG files in the file picker
#   multiple=False  → allow only one file at a time
#
# After running this cell:
#   1. Click the "Upload" button that appears
#   2. Select a PNG image from your computer
#   3. Continue to the next cell — the uploaded data is stored
#      in uploader.value
#
# In production apps you would read images from a URL, a database,
# or a file path — this widget is just convenient for notebooks.

from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [42]:
# ============================================================
# CELL 5: Inspect the Raw Uploaded File Data
# ============================================================
# uploader.value is a tuple of dicts — one per uploaded file.
# Each dict has:
#   'name'     → original filename (e.g. 'city.png')
#   'type'     → MIME type (e.g. 'image/png')
#   'size'     → file size in bytes
#   'content'  → a memoryview of the raw binary bytes
#
# A memoryview is a Python object that gives zero-copy access
# to a buffer. We need to convert it to bytes before base64 encoding.

print(uploader.value)

({'name': 'moon.png', 'type': 'image/png', 'size': 358916, 'content': <memory at 0x00000283375C4A00>, 'last_modified': datetime.datetime(2026, 6, 30, 13, 1, 19, 250000, tzinfo=datetime.timezone.utc)},)


In [43]:
# ============================================================
# CELL 6: Convert the Image to a Base64 String
# ============================================================
# Step by step:
#
# 1. uploader.value[0]
#    → The first (and only) uploaded file dict
#
# 2. uploaded_file["content"]
#    → The raw binary data as a memoryview (like a pointer to bytes)
#
# 3. bytes(content_mv)  or  content_mv.tobytes()
#    → Converts memoryview to a regular Python bytes object
#    → Required because base64.b64encode() needs bytes, not memoryview
#
# 4. base64.b64encode(img_bytes)
#    → Encodes binary bytes as base64 bytes (still bytes, not string)
#
# 5. .decode("utf-8")
#    → Converts base64 bytes to a Python string
#    → Now safe to embed in a JSON dict / HTTP request
#
# img_b64 is the string we pass to the model.

import base64

uploaded_file = uploader.value[0]          # Get the first uploaded file
content_mv = uploaded_file["content"]      # memoryview of raw bytes
img_bytes = bytes(content_mv)              # Convert to regular bytes
img_b64 = base64.b64encode(img_bytes).decode("utf-8")  # Base64 string


In [46]:
# ============================================================
# CELL 7: Send Text + Image Together to the Vision Model
# ============================================================
# Now we build a multimodal content list with TWO blocks:
#   Block 1: text instruction  →  what to do with the image
#   Block 2: the image itself  →  base64-encoded PNG
#
# The image block fields:
#   "type"      → must be "image"
#   "base64"    → the base64-encoded image string from Cell 6
#   "mime_type" → the image format ("image/png", "image/jpeg", etc.)
#                 The model needs this to decode the image correctly.
#
# The model receives BOTH blocks and reasons across them together:
#   it reads the text instruction AND looks at the image simultaneously.
#
# 💡 Tip: put the text instruction BEFORE the image block — it gives
#    the model framing before it processes the visual content.
from langchain.messages import HumanMessage
multimodal_question = HumanMessage(content=[
    {"type": "text",  "text": "Tell me about this capital"},    # Instruction first
    {
        "type": "image_url",
        "image_url": {
            "url": f"data:image/png;base64,{img_b64}"
        }  # Then the image
    }
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)


<think>
The user wants a description of a capital city based on the provided image.

**1. Visual Analysis:**
*   **Setting:** A barren, rocky, desert-like landscape. It looks like a moon or a terraformed planet. The ground is brown/orange, dusty, and rugged.
*   **Sky:** Deep blue, almost teal, indicating a thin atmosphere or a specific time of day (dusk/dawn) or just a different atmospheric composition.
*   **Celestial Bodies:** A massive, cratered moon dominates the sky. It's very close, suggesting a binary planet system or a very large moon. There are other smaller moons or planets visible in the distance.
*   **The City:**
    *   **Architecture:** Tall, sleek, needle-like spires. They look metallic, silver/grey with glowing blue accents. They are clustered together on a rocky plateau or ridge.
    *   **Style:** Futuristic, sci-fi, perhaps "clean" or "sterile" looking. Not organic or bulky.
    *   **Location:** Perched high up, overlooking a valley.
    *   **Key Feature:** In t

---
## Section 3 — Audio Input

Audio follows the same pattern as images:
1. Record or load audio → get raw bytes
2. Base64 encode
3. Include in a content block with `type: "audio"`

**Important:** not all models support audio. You must use a model specifically built for audio input, like `gpt-4o-audio-preview`. Using the wrong model will raise an error.

### Audio recording pipeline

```
sounddevice.rec()  →  numpy array (raw audio samples)
scipy.io.wavfile.write()  →  WAV file bytes in memory
base64.b64encode()  →  base64 string
HumanMessage(content=[{"type": "audio", ...}])  →  model
```

In [55]:
# ============================================================
# CELL 8: Record Audio from Your Microphone
# ============================================================
# Libraries used:
#   sounddevice  → cross-platform microphone access
#   scipy        → write numpy audio array to WAV format
#   tqdm         → progress bar so you know how long is left
#   io.BytesIO   → in-memory file buffer (no disk write needed)
#
# Recording settings:
#   duration=5       → record for 5 seconds
#   sample_rate=44100→ CD quality (44,100 samples per second)
#                      This is standard; lower values (16000) work
#                      fine for speech and use less bandwidth.
#   channels=1       → mono (stereo is 2 channels; mono is enough
#                      for speech and is half the data)
#
# sd.rec() starts recording in the background (non-blocking).
# The tqdm loop counts down the duration visually.
# sd.wait() blocks until recording is fully complete.
#
# BytesIO is a "fake file" in RAM — scipy writes the WAV bytes into it
# just like writing to a disk file, but nothing touches the filesystem.
# .getvalue() reads all the bytes out of the buffer.

import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

duration = 5        # seconds to record
sample_rate = 44100 # samples per second (CD quality)

print("Recording...")
audio = sd.rec(
    int(duration * sample_rate),
    # Total number of samples to record.
    # duration (in seconds) × sample_rate (samples per second) = total samples.

    samplerate=sample_rate,
    # Sample rate = number of audio samples captured per second (Hz).
    # Example: 44100 Hz (CD quality) means 44,100 samples are recorded every second.
    # Higher sample rate → better audio quality but larger file size.

    channels=1
    # Number of audio channels:
    # 1 = Mono (single channel, suitable for speech recording)
    # 2 = Stereo (two channels, left + right, used for music or spatial audio)
)

# Show a progress bar while the recording happens
for _ in tqdm(range(duration * 10)):  # Tick 10 times per second
    time.sleep(0.1)

sd.wait()  # Block until recording finishes
print("Done.")

# Write WAV audio to an in-memory buffer (no disk I/O)
buf = io.BytesIO()
write(buf, sample_rate, audio)  # Writes WAV-formatted bytes into buf
wav_bytes = buf.getvalue()      # Read all bytes from the buffer

# Base64 encode for JSON transport
aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.77it/s]


Done.


In [56]:
print(aud_b64)

...


In [57]:
import os
import base64
import tempfile

from groq import Groq
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

client = Groq()

# ------------------------------------------------------------
# Decode the base64 audio into a temporary WAV file
# ------------------------------------------------------------
tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
try:
    tmp.write(base64.b64decode(aud_b64))
    tmp.close()

    with open(tmp.name, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-large-v3",
            file=audio_file,
        )
finally:
    os.remove(tmp.name)
print("Transcript:")
print(transcript.text)

# ------------------------------------------------------------
# Ask GPT-5 to analyze the transcript
# ------------------------------------------------------------
agent = create_agent(model="groq:openai/gpt-oss-20b")

response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=f"""
Tell me about this audio.

Transcript:
{transcript.text}

Please provide:
1. A summary
2. Main topic
3. Sentiment
4. Important observations
"""
            )
        ]
    }
)

print(response["messages"][-1].content)

Transcript:
 నాయ్ మా పేరు యోగవర్ధన్ రేడ్డి మేరోక ప్రోగ్రామర్నే
**Transcript (Telugu)**  
“నాయ్ మా పేరు యోగవర్ధన్ రేడ్డి మేరోక ప్రోగ్రామర్నే”  
(Approximate translation: “My name is Yoganvardhan Reddy, I am a programmer.”)

---

### 1. Summary  
The speaker gives a brief self‑introduction, stating his name (Yoganvardhan Reddy) and that he works as a programmer.

### 2. Main Topic  
- Personal introduction / self‑identification  
- Professional identity (programmer)

### 3. Sentiment  
Neutral. The statement is factual and straightforward, with no overt emotional content.

### 4. Important Observations  
- **Speaker’s Profession**: Identifies himself as a programmer, indicating technical background.  
- **Language & Region**: The use of Telugu suggests the speaker is from a Telugu‑speaking community.  
- **Length & Context**: The audio is very short and likely part of a larger introduction or introductory segment in a video/podcast.  
- **Pronunciation Note**: The phrase “మేరోక” may be a

---
## Summary

| Concept | Key takeaway |
|---|---|
| **Content list format** | `HumanMessage(content=[...])` — replaces plain string for multimodal messages |
| **Content block** | A dict with `type` + modality-specific fields (`text`, `base64`, `mime_type`) |
| **Base64** | Converts binary files (PNG, WAV) into ASCII strings safe for JSON transport |
| **`memoryview → bytes`** | Jupyter widgets return `memoryview` — convert with `bytes(mv)` before base64 |
| **`BytesIO`** | In-memory fake file — write audio/image bytes without touching the disk |
| **Model matters** | Audio requires `gpt-4o-audio-preview` — check modality support before switching models |

### The three content block types

```python
# Text
{"type": "text", "text": "your message"}

# Image
{"type": "image", "base64": img_b64, "mime_type": "image/png"}

# Audio
{"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
```

### Next steps
- **1.5** — Personal Chef: putting tools + memory + prompting together into a real app
- Explore adding image understanding to the chef app (e.g. "what can I make with these ingredients?" + photo of a fridge)